In [3]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [4]:
bases_utilizadas = 'bases_utilizadas/'
arquivos_gerados = 'arquivos_gerados/'

# Análise
Com o dataframe `distancia_residencia_pericias_NE.csv`, na pasta `arquivos_gerados`, é possível começar a partir desta etapa.

As perícias serão classificadas em duas dimensões:

I. Tipo de distância: mesma cidade (distância igual a zero) e cidade diferente;

II. Faixa de distância: cidade a “até 70 km” ou “mais de 70 km”, incluindo a categoria “mesma cidade”.

Para o cálculo de quantos requerentes precisaram tiveram procedimentos marcados em cada faixa de distância, foi adotado como critério a maior distância entre a residência e a APS para cada requerente e em cada ano, para evitar dupla contagem de indivíduos que possam ter requisitaddo mais de uma perícia para locais a diferentes distâncias do município de residência.

In [5]:
df_pericias = pd.read_csv(arquivos_gerados + 'distancia_residencia_pericias_NE.csv')

In [6]:
df_pericias.info()

<class 'pandas.DataFrame'>
RangeIndex: 697230 entries, 0 to 697229
Data columns (total 21 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   Identificação anonimizada do Requerente  697230 non-null  str    
 1   CD_MUN_res                               697230 non-null  int64  
 2   NM_MUN_res                               697230 non-null  str    
 3   SIGLA_UF_res                             697230 non-null  str    
 4   lon_res                                  697230 non-null  float64
 5   lat_res                                  697230 non-null  float64
 6   CD_MUN_aps                               697230 non-null  int64  
 7   NM_MUN_aps                               697230 non-null  str    
 8   SIGLA_UF_aps                             697230 non-null  str    
 9   lon_aps                                  697230 non-null  float64
 10  lat_aps                                  69

### Questionamento 1

Quantos requerentes tiveram perícias marcadas em cidades a mais de 70km do município de residência em 2025?

In [7]:
df_pericias

,Identificação anonimizada do Requerente,CD_MUN_res,NM_MUN_res,SIGLA_UF_res,lon_res,lat_res,CD_MUN_aps,NM_MUN_aps,SIGLA_UF_aps,lon_aps,...,2019,2020,2021,2022,2023,2024,2025,2026,TOTAL,dist_km
0,Requerente 1,2309607,Pacajus,CE,-38.499413,-4.183950,2304400,Fortaleza,CE,-38.529089,...,0,0,1,0,0,0,0,0,1,43.596913
1,Requerente 2,2311405,Quixeramobim,CE,-39.333549,-5.186537,2304400,Fortaleza,CE,-38.529089,...,0,0,0,0,0,1,0,0,1,178.782997
2,Requerente 3,2302602,Camocim,CE,-40.790527,-2.966912,2302602,Camocim,CE,-40.790527,...,0,0,0,0,0,0,1,0,1,0.000000
3,Requerente 4,2302602,Camocim,CE,-40.790527,-2.966912,2306553,Itarema,CE,-39.879588,...,0,0,0,0,0,0,1,0,1,101.793160
4,Requerente 5,2314102,Viçosa do Ceará,CE,-41.118011,-3.503042,2304400,Fortaleza,CE,-38.529089,...,0,0,0,0,0,0,1,0,1,289.095167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697225,Requerente 582050,2312908,Sobral,CE,-40.044392,-3.856256,2211001,Teresina,PI,-42.699230,...,0,0,0,0,0,0,1,0,1,329.269179
697226,Requerente 582051,2309706,Pacatuba,CE,-38.597428,-3.953830,2304400,Fortaleza,CE,-38.529089,...,0,0,0,0,1,1,0,0,2,19.424928
697227,Requerente 582052,2304400,Fortaleza,CE,-38.529089,-3.792993,2304400,Fortaleza,CE,-38.529089,...,0,0,0,1,1,0,0,0,2,0.000000
697228,Requerente 582053,2307403,Jucás,CE,-39.590693,-6.468283,2305506,Iguatu,CE,-39.282929,...,0,0,0,0,0,0,0,1,1,36.380368


In [8]:
df_distancia = (df_pericias
                .melt(id_vars=['Identificação anonimizada do Requerente', 'NM_MUN_res', 'NM_MUN_aps', 'dist_km'],
                      value_vars=['2019','2020','2021','2022','2023','2024','2025','2026', 'TOTAL'],
                      var_name='Ano',
                      value_name='Quantidade'))

filtro_anos = ['2019','2020','2021','2022','2023','2024','2025']

df_distancia = df_distancia[(df_distancia['Quantidade'] > 0) & (df_distancia['Ano'].isin(filtro_anos))]

In [9]:
# Classificações das distâncias

df_max = (
    df_distancia
    .groupby(['Ano', 'Identificação anonimizada do Requerente'])
    .agg({'dist_km': 'max'})
    .reset_index()
)

df_max['faixa_dist'] = np.select(
    [
        df_max['dist_km'] == 0,
        df_max['dist_km'] > 70
    ],
    [
        'Mesma cidade',
        'Mais de 70 km'
    ],
    default='Até 70 km'
)

df_max['tipo_distancia'] = np.where(
    df_max['dist_km'] == 0,
    'Mesma cidade',
    'Cidade diferente'
)

In [10]:
# Requerentes por TIPO DE DISTÂNCIA

# Seleciona as maiores distâncias entre município de residência e da APS por requerente e ano
df_unico = (
    df_max
    .sort_values(['Ano', 'Identificação anonimizada do Requerente', 'dist_km'],
                 ascending=[True, True, False])
    .drop_duplicates(subset=['Ano', 'Identificação anonimizada do Requerente'],
                     keep='first')
)

df_tipo_distancia = (
    df_unico
    .groupby(['Ano', 'tipo_distancia'])['Identificação anonimizada do Requerente']
    .nunique()
    .reset_index()
)

df_tipo_distancia = df_tipo_distancia.pivot(
    index='Ano',
    columns='tipo_distancia',
    values='Identificação anonimizada do Requerente'
).fillna(0)

# Total e proporção
df_tipo_distancia['Total'] = df_tipo_distancia.sum(axis=1)

df_prop = round(
    df_tipo_distancia.drop(columns='Total').div(df_tipo_distancia['Total'], axis=0) * 100,
    2
)

# Dataframe final para o tipo de distância
df_final_tipo_distancia = df_tipo_distancia.merge(
    df_prop,
    left_index=True,
    right_index=True,
    suffixes=('', ' (%)')
).reset_index()

In [11]:
df_final_tipo_distancia

tipo_distancia,Ano,Cidade diferente,Mesma cidade,Total,Cidade diferente (%),Mesma cidade (%)
0,2019,1081,1533,2614,41.35,58.65
1,2020,26360,28895,55255,47.71,52.29
2,2021,58523,47485,106008,55.21,44.79
3,2022,71557,52320,123877,57.76,42.24
4,2023,53547,40916,94463,56.69,43.31
5,2024,92977,51183,144160,64.50,35.50
6,2025,156371,75041,231412,67.57,32.43


In [12]:
round(df_final_tipo_distancia.set_index('Ano').pct_change(1) * 100,2)

tipo_distancia,Cidade diferente,Mesma cidade,Total,Cidade diferente (%),Mesma cidade (%)
Ano,,,,,
2019,NaN,NaN,NaN,NaN,NaN
2020,2338.48,1784.87,2013.81,15.38,-10.84
2021,122.01,64.34,91.85,15.72,-14.34
2022,22.27,10.18,16.86,4.62,-5.69
2023,-25.17,-21.80,-23.74,-1.85,2.53
2024,73.64,25.09,52.61,13.78,-18.03
2025,68.18,46.61,60.52,4.76,-8.65


In [13]:
df_final_tipo_distancia.to_csv(arquivos_gerados + 'tipo_distancia_pericias.csv', index=False, encoding='UTF-8')

In [14]:
# Requerentes por FAIXA DE DISTÂNCIA

df_faixa_distancia = (
    df_max[df_max['faixa_dist'] != 'Mesma cidade']
    [['Ano', 'faixa_dist', 'Identificação anonimizada do Requerente']]
    .groupby(['Ano', 'faixa_dist'])
    .nunique()
    .reset_index()
    )

df_faixa_distancia = (
    df_faixa_distancia.pivot(
        index='Ano',
        columns='faixa_dist',
        values='Identificação anonimizada do Requerente')
).fillna(0)

# Total e proporção
df_faixa_distancia['Total'] = df_faixa_distancia.sum(axis=1)

df_prop = round(
    df_faixa_distancia.drop(columns='Total').div(df_faixa_distancia['Total'], axis=0) * 100,
    2
)

# Dataframe final para o tipo de deslocamento
df_final_faixa_distancia = df_faixa_distancia.merge(
    df_prop,
    left_index=True,
    right_index=True,
    suffixes=('', ' (%)')
).reset_index()

In [15]:
df_final_faixa_distancia.to_csv(arquivos_gerados + 'faixa_distancia_pericias.csv', index=False, encoding='UTF-8')

In [16]:
df_final_faixa_distancia

faixa_dist,Ano,Até 70 km,Mais de 70 km,Total,Até 70 km (%),Mais de 70 km (%)
0,2019,640,441,1081,59.20,40.80
1,2020,15506,10854,26360,58.82,41.18
2,2021,28034,30489,58523,47.90,52.10
3,2022,34593,36964,71557,48.34,51.66
4,2023,26267,27280,53547,49.05,50.95
5,2024,46598,46379,92977,50.12,49.88
6,2025,80662,75709,156371,51.58,48.42


In [17]:
round(df_final_faixa_distancia.set_index('Ano').pct_change(1) * 100,2)

faixa_dist,Até 70 km,Mais de 70 km,Total,Até 70 km (%),Mais de 70 km (%)
Ano,,,,,
2019,NaN,NaN,NaN,NaN,NaN
2020,2322.81,2361.22,2338.48,-0.64,0.93
2021,80.79,180.90,122.01,-18.57,26.52
2022,23.40,21.24,22.27,0.92,-0.84
2023,-24.07,-26.20,-25.17,1.47,-1.37
2024,77.40,70.01,73.64,2.18,-2.10
2025,73.10,63.24,68.18,2.91,-2.93


### Questionamento 2

Quais as "origens x destinos" mais frequentes? Critério utilizado: requerentes que fizeram o trajeto, por ano.

In [18]:
df_origem_destino = df_pericias[['Identificação anonimizada do Requerente', 'NM_MUN_res',
                                 'SIGLA_UF_res', 'NM_MUN_aps', 'SIGLA_UF_aps', 'dist_km',
                                 '2019', '2020', '2021', '2022', '2023', '2024', '2025']]
df_origem_destino = df_origem_destino[(df_origem_destino['dist_km'] > 0)]

In [19]:
df_origem_destino_melt = df_origem_destino.melt(
    id_vars=['Identificação anonimizada do Requerente', 'NM_MUN_res',
             'SIGLA_UF_res', 'NM_MUN_aps', 'SIGLA_UF_aps', 'dist_km'],
    value_vars=['2019','2020','2021','2022','2023','2024','2025'],
            var_name='Ano', value_name='Quantidade')
             

df_origem_destino_melt = df_origem_destino_melt[df_origem_destino_melt['Quantidade'] > 0]

df_origem_destino_melt

,Identificação anonimizada do Requerente,NM_MUN_res,SIGLA_UF_res,NM_MUN_aps,SIGLA_UF_aps,dist_km,Ano,Quantidade
224,Requerente 292,Aracati,CE,Fortaleza,CE,136.279584,2019,1
318,Requerente 399,Maracanaú,CE,Fortaleza,CE,15.209933,2019,1
748,Requerente 920,Caucaia,CE,Fortaleza,CE,29.303198,2019,1
902,Requerente 1089,Caucaia,CE,Fortaleza,CE,29.303198,2019,1
1553,Requerente 1909,Horizonte,CE,Fortaleza,CE,34.095438,2019,1
...,...,...,...,...,...,...,...,...
3152710,Requerente 582035,Parambu,CE,Picos,PI,121.870429,2025,1
3152711,Requerente 582036,Parambu,CE,Fortaleza,CE,367.406775,2025,1
3152715,Requerente 582039,Limoeiro do Norte,CE,Mossoró,RN,82.515432,2025,1
3152717,Requerente 582044,Fortaleza,CE,Tianguá,CE,270.246466,2025,1


In [20]:
# Por total de perícicas
df_origem_destino_melt[['Ano', 'NM_MUN_res', 'SIGLA_UF_res', 'NM_MUN_aps',
                        'SIGLA_UF_aps', 'dist_km', 'Quantidade']].groupby(
                            ['Ano', 'NM_MUN_res', 'SIGLA_UF_res', 'NM_MUN_aps',
                             'SIGLA_UF_aps', 'dist_km']
                        ).sum().reset_index().sort_values(by=['Ano', 'Quantidade'])

,Ano,NM_MUN_res,SIGLA_UF_res,NM_MUN_aps,SIGLA_UF_aps,dist_km,Quantidade
0,2019,Abaiara,CE,Brejo Santo,CE,29.796720,1
3,2019,Acaraú,CE,Fortaleza,CE,186.397702,1
5,2019,Acopiara,CE,Crato,CE,126.576908,1
7,2019,Aiuaba,CE,Crato,CE,120.364992,1
8,2019,Alcântaras,CE,Sobral,CE,63.727169,1
...,...,...,...,...,...,...,...
13514,2025,Juazeiro do Norte,CE,Crato,CE,24.904557,1618
14943,2025,Quixadá,CE,Fortaleza,CE,136.680590,1959
13769,2025,Maranguape,CE,Fortaleza,CE,32.846857,2309
13713,2025,Maracanaú,CE,Fortaleza,CE,15.209933,4469


In [21]:
# Por requerentes
df_origem_destino_requerentes = (
    df_origem_destino_melt[['Ano', 'NM_MUN_res', 'SIGLA_UF_res', 'NM_MUN_aps', 'SIGLA_UF_aps', 'dist_km', 'Identificação anonimizada do Requerente']]
    .groupby(['Ano', 'NM_MUN_res', 'SIGLA_UF_res', 'NM_MUN_aps', 'SIGLA_UF_aps', 'dist_km'])
    .nunique()
    .reset_index()
    .sort_values(by=['Ano', 'Identificação anonimizada do Requerente']))

In [22]:
df_origem_destino_requerentes['Origem x Destino'] = (df_origem_destino_requerentes['NM_MUN_res'] + ' (' +
  df_origem_destino_requerentes['SIGLA_UF_res'] + ') ' + ' ⮕ ' + 
  df_origem_destino_requerentes['NM_MUN_aps'] + ' (' +
  df_origem_destino_requerentes['SIGLA_UF_aps'] + ')')

In [23]:
lista_rank = []
for ano in df_origem_destino_requerentes['Ano'].unique():
    rank = (df_origem_destino_requerentes[df_origem_destino_requerentes['Ano'] == ano]
            [['Ano', 'Origem x Destino', 'NM_MUN_res', 'NM_MUN_aps', 'Identificação anonimizada do Requerente', 'dist_km']]
            .sort_values(['Identificação anonimizada do Requerente'],
                         ascending=False)).head(10)
    lista_rank.append(rank)

df_rank_origem_destino = pd.concat(lista_rank).sort_values(['Ano', 'Identificação anonimizada do Requerente'], ascending=[False, False])

# Inserir coluna com a faixa da distância
df_rank_origem_destino['faixa_dist'] = np.select(
    [
        df_rank_origem_destino['dist_km'] > 70
    ],
    [
        'Mais de 70 km'
    ],
    default='Até 70 km'
)

In [24]:
df_rank_origem_destino.to_csv(arquivos_gerados + 'df_rank_origem_destino.csv', index=False)

### Questionamento 3

Em quais cidades moram os requerentes que tiveram as perícias marcadas para mais longe de casa?

In [25]:
df_media_ponderada = df_pericias.copy()

df_media_ponderada = (df_media_ponderada
                      .melt(
                          id_vars=['Identificação anonimizada do Requerente', 'NM_MUN_res', 'dist_km'],
                          value_vars=['2019', '2020', '2021', '2022', '2023', '2024', '2025'],
                          var_name='Ano',
                          value_name='total_pericias'
                          ))

In [26]:
df_media_ponderada = df_media_ponderada[
    (df_media_ponderada['dist_km'] != 0) & 
    (df_media_ponderada['total_pericias'] != 0)
]

# cálculo da média ponderada

df_media_ponderada_distancia = (
    df_media_ponderada
    .assign(dist_pond = df_media_ponderada['dist_km'] * df_media_ponderada['total_pericias'])
    .groupby(['Ano', 'NM_MUN_res'])
    .agg(
        soma_dist_pond = ('dist_pond', 'sum'),
        total_pericias = ('total_pericias', 'sum')
    )
    .assign(dist_media_km = lambda x: x['soma_dist_pond'] / x['total_pericias'])
    .reset_index()
)

# Arredondar média e selecionar colunas
df_media_ponderada_distancia['dist_media_km'] = df_media_ponderada_distancia['dist_media_km'].round(2)
df_media_ponderada_distancia = df_media_ponderada_distancia[
    ['Ano', 'NM_MUN_res', 'dist_media_km', 'total_pericias']
]

In [27]:
# As 10 maiores distâncias médias por ano

lista_media_distancia = []
for ano in df_media_ponderada_distancia['Ano'].unique():
    top10 = df_media_ponderada_distancia[df_media_ponderada_distancia['Ano'] == ano].sort_values(by=['Ano', 'dist_media_km'], ascending=[False, False]).head(10)
    lista_media_distancia.append(top10)

df_final_media_ponderada = pd.concat(lista_media_distancia)

df_final_media_ponderada.rename(columns={
    'NM_MUN_res': 'Município de residência',
    'dist_media_km': 'Distância média',
    'total_pericias': 'Total de perícias'
}, inplace=True)

In [28]:
df_final_media_ponderada.to_csv(arquivos_gerados + 'df_final_media_ponderada.csv', index=False)